In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")

cornwall_granular_collection = Chroma(collection_name="cornwall_granular",
    embedding_function=embeddings_model)

cornwall_granular_collection.reset_collection()

cornwall_coarse_collection = Chroma(collection_name="cornwall_coarse",
    embedding_function=embeddings_model)
cornwall_coarse_collection.reset_collection()

/Users/bahloulia/Downloads/agentic_software/RAG-Applications/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_community.document_loaders import AsyncHtmlLoader

# Wikimedia's crawler policy blocks/serves a robots-notice page instead of
# the article for requests with no identifying User-Agent - same fix as
# the wikipedia loader in the other notebook.
wikimedia_header_template = {
    "User-Agent": "RAG-Applications/1.0 (amine.bahlouli1993@gmail.com)"}

destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(
    destination_url, header_template=wikimedia_header_template)
docs = html_loader.load()
from langchain_text_splitters import HTMLSectionSplitter

headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(
            html_string)
        all_chunks.extend(temp_chunks) 

    return all_chunks

granular_chunks = split_docs_into_granular_chunks(docs)
cornwall_granular_collection.add_documents(documents=granular_chunks)
results = cornwall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

In [9]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [ ]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import sys
import uuid

sys.path.append("..")
from llm_model import llm

# TODO: fill in the actual set of UK destination pages you want to index
uk_destination_urls = [
    "https://en.wikivoyage.org/wiki/Cornwall",
    "https://en.wikivoyage.org/wiki/London",
    "https://en.wikivoyage.org/wiki/Edinburgh",
    "https://en.wikivoyage.org/wiki/Bath",
]

# Wikimedia's crawler policy blocks/serves a robots-notice page instead of
# the article for requests with no identifying User-Agent.
wikimedia_header_template = {
    "User-Agent": "RAG-Applications/1.0 (amine.bahlouli1993@gmail.com)"}

html2text_transformer = Html2TextTransformer()

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000)

summaries_collection = Chroma(
    collection_name="uk_summaries",
    embedding_function=embeddings_model,
)

summaries_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)



summarization_chain = (
    {"document": lambda x: x.page_content}
    | ChatPromptTemplate.from_template(
        "Summarize the following document:\n\n{document}")
    | llm
    | StrOutputParser())

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url, header_template=wikimedia_header_template)
    html_docs =  html_loader.load()
    text_docs = html2text_transformer.transform_documents(
        html_docs)

    coarse_chunks = parent_splitter.split_documents(
        text_docs)

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks):

        coarse_chunk_id = coarse_chunks_ids[i]

        summary_text =  summarization_chain.invoke(
            coarse_chunk)
        summary_doc = Document(page_content=summary_text, 
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries)
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks)))

retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

print(summary_docs_only[0])

In [3]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")
print(retrieved_docs[0])

page_content='The Night Riviera sleeper service operates between London and Penzance six
nights a week (Su-F). Trains feature cabins and an on-board lounge.

Great Western Railway also operates regular slow trains between Bristol Temple
Meads, Exeter St Davids, Plymouth and Penzance, calling in Cornwall at
**Saltash** , **St Germans** , **Liskeard** , **Bodmin Parkway** ,
**Lostwithiel** , **Par** , **St Austell** , **Truro** , **Redruth** ,
**Camborne** , **Hayle** , **St Erth** and **Penzance**.

Some destinations lie on local lines, so journeys may require a change of
trains at:

  * Plymouth for **Calstock** and **Gunnislake**
  * Liskeard for **Looe**
  * Par for **Newquay**
  * Truro for **Penryn** and **Falmouth**
  * St Erth or Penzance for **St Ives**

A small number of CrossCountry services call at **Liskeard** , **Bodmin
Parkway** , **St Austell** , **Truro** , **Redruth** , **St Erth** and
**Penzance** from destinations throughout the UK, including: Edinburgh,
Newcastle upo